# C# and SQLite

```{contents}
:local:
:depth: 2
```

Use a saved .NET console project for this workflow. It creates files and uses a provider package, so complete it in VS Code rather than a browser code cell. The project uses classes, nullable results, collections, exceptions, and `using` declarations. Review those core chapters before starting.


## Set Up and Run

```{index} Set Up and Run
```


Download or clone the book repository and open `demos/_extras/bookstore_inventory/`. Its project file pins the SQLite provider version used by the reference implementation.

```sh
cd demos/_extras/bookstore_inventory
dotnet restore
dotnet run
dotnet run -- --check
```

The first restore requires internet access. No database server or account is needed. Run from the project directory so the relative database path is predictable.

The ordinary run creates `bookstore.db` and lists two seeded titles. Later runs preserve data; seed IDs are inserted only if absent. `--check` uses a new in-memory database, leaving your saved inventory untouched. To reset practice data, stop the program and delete only your local `bookstore.db`, then run again.

- [Project file](../../../demos/_extras/bookstore_inventory/BookstoreInventory.csproj)
- [Complete reference source](../../../demos/_extras/bookstore_inventory/Program.cs)
- {download}`Project instructions <../../../demos/_extras/bookstore_inventory/README.md>`


## Connections, Commands, and Disposal

```{index} Connections, Commands, and Disposal
```


A `SqliteConnection` opens the local database. A command holds SQL text and parameters. A reader steps through result rows. `using` disposes each resource when its scope ends, including when an exception leaves the scope. This `using` declaration manages a resource; it has a different role from a namespace import.

The reference project follows this order: open a connection, enable foreign keys, create tables, seed missing sample records, construct an `Inventory`, and request operations. Inspect the setup before the class definitions in `Program.cs`.


The following complete `Program.cs` demonstrates one connection, setup command, parameter, and reader. Run it in a separate console project with the same SQLite package as the supplied project. Its in-memory database is recreated each run. Expected output: `C# for Business`.

```csharp
using Microsoft.Data.Sqlite;

using var connection = new SqliteConnection("Data Source=:memory:");
connection.Open();
using (var setup = connection.CreateCommand())
{
    setup.CommandText = """
        CREATE TABLE Books (BookId INTEGER PRIMARY KEY, Title TEXT NOT NULL);
        INSERT INTO Books VALUES (1, 'C# for Business');
        """;
    setup.ExecuteNonQuery();
}
using var query = connection.CreateCommand();
query.CommandText = "SELECT Title FROM Books WHERE BookId = @id;";
query.Parameters.AddWithValue("@id", 1);
using var reader = query.ExecuteReader();
while (reader.Read())
    Console.WriteLine(reader.GetString(0));
```


## Map Rows to Objects

```{index} Map Rows to Objects
```


`Book` groups an ID, title, price in cents, and stock. `Inventory.List` uses an explicit column order and reads each row into a new `Book`. `Find` returns either one book or `null`; missing data is an expected result, not automatically an exception.

Read `ReadBook` and match every reader position to its selected column. Predict what could go wrong if a query changed column order without updating the mapper. Keep each command's result shape explicit.


```{index} SQLite; row mapping; affected rows
```

### Follow a Row from SQL to Local Values


The column positions in a reader follow the `SELECT` list, not the order you happen to think of the object's fields. This miniature example deliberately uses only ID, title and stock. Run the complete listing as `Program.cs` in a separate project with the same SQLite package; its in-memory database starts fresh. It does not replace the full inventory model.

General mapping pattern (schematic):

```csharp
using var reader = command.ExecuteReader();
if (reader.Read())
{
    var first = reader.GetInt64(0);    // match the selected column and its type
    var second = reader.GetString(1);
    // construct the object from these values
}
```

`Read()` advances to a row and reports whether one exists. Do not read fields before that succeeds. The zero-based positions identify columns in that row. The appropriate getter interprets the stored value as the expected .NET type; it does not repair a mismatched column list.


```csharp
using System;
using Microsoft.Data.Sqlite;
using var connection = new SqliteConnection("Data Source=:memory:");
connection.Open();
using (var setup = connection.CreateCommand())
{
    setup.CommandText = "CREATE TABLE Books (BookId INTEGER PRIMARY KEY, Title TEXT, Stock INTEGER); INSERT INTO Books VALUES (1, 'C# Operations', 6);";
    setup.ExecuteNonQuery();
}
using (var query = connection.CreateCommand())
{
    query.CommandText = "SELECT BookId, Title, Stock FROM Books WHERE BookId = @id;";
    query.Parameters.AddWithValue("@id", 1);
    using var reader = query.ExecuteReader();
    if (reader.Read())
    {
        long id = reader.GetInt64(0);
        string title = reader.GetString(1);
        int stock = reader.GetInt32(2);
        Console.WriteLine($"{id}: {title}, stock={stock}");
    }
}
using var update = connection.CreateCommand();
update.CommandText = "UPDATE Books SET Stock = @stock WHERE BookId = @id;";
update.Parameters.AddWithValue("@stock", 4);
update.Parameters.AddWithValue("@id", 999);
Console.WriteLine($"unknown ID affected rows={update.ExecuteNonQuery()}");
```


The output is `1: C# Operations, stock=6`, then `unknown ID affected rows=0`. Position 0 maps to `BookId`, position 1 to `Title`, and position 2 to `Stock`. In the full project, `ReadBook` adds `PriceCents` and constructs an object from four columns using the same rule. Reordering the query to `Title, BookId, Stock` without changing these getters makes the mapper wrong; adding more casts does not fix the contract.

An update's affected-row count is a different result from a selected record. Here ID 999 is absent, so the command changes zero rows. In this simple schema, updating an existing primary key changes one matching row; the supplied `SetStock` converts that count into a success Boolean.


```{index} injection; SQL injection
```

## Injection

**Injection** happens when a program builds a command, such as a database query, by gluing user input into the command text. Input crafted by an attacker then becomes part of the command:

In [1]:
using System;

// Building a database query by gluing user input into the text is dangerous.
static string UnsafeQuery(string customerName)
{
    return "SELECT * FROM Customers WHERE Name = '" + customerName + "'";
}

Console.WriteLine(UnsafeQuery("Ana Diaz"));
Console.WriteLine(UnsafeQuery("x' OR '1'='1"));   // now matches every customer

SELECT * FROM Customers WHERE Name = 'Ana Diaz'
SELECT * FROM Customers WHERE Name = 'x' OR '1'='1'


The second query treats supplied text as part of SQL syntax. This example builds strings only; it does not execute them. Use the parameterized commands taught in this appendix to pass values separately from command text. The input is then treated as data rather than SQL syntax.


## Supply Parameters and Check Outcomes

```{index} Supply Parameters and Check Outcomes
```


`Add` passes the title, price, and stock as values. A title such as `Manager's SQL Guide` needs no manual quote replacement. `SetStock` uses an ID condition and reports whether exactly one row changed. Reject invalid quantities before sending the command, and keep database constraints as a second boundary.

In the source, compare `Find` with `SetStock`: both use an ID parameter, but one reads a row and the other checks the affected-row count. Explain the expected result for an unknown ID in each operation.


## Transactions for Related Changes

```{index} Transactions for Related Changes
```


The optional `Sell` operation begins a transaction, reduces stock only if enough remains, inserts a sale, and commits. If the book is missing or stock is insufficient, it rolls back and returns `false`. If an unexpected error occurs before commit, disposal rolls back the uncommitted transaction.

Every command participating in the transaction sets its `Transaction` property. Updating stock and inserting the sale independently could leave only one change saved after a failure. Atomicity means these changes are committed together.


```{index} transactions; rollback; atomicity
```

### Trace a Sale That Fails After the Stock Update


The transaction's purpose is to treat related persistent changes as one unit. The schematic pattern is:

```csharp
using var transaction = connection.BeginTransaction();
// Every participating command sets command.Transaction = transaction.
// Execute and check the related changes.
transaction.Commit();
```

Commit makes the unit permanent. An explicit rollback, or disposal without a successful commit, abandons its changes. Do not catch a database failure and report a successful sale. A valid row-level constraint is useful, but it cannot by itself coordinate two separate commands.

| Step | Stock visible inside transaction | Sale rows | Meaning |
| --- | ---: | ---: | --- |
| Before sale | 6 | 0 | starting state |
| Decrement by two | 4 | 0 | pending, not yet committed |
| Sale insert fails | 4 | 0 | incomplete unit |
| Rollback completes | 6 | 0 | original state restored |

Run this complete listing in a separate SQLite console project. The invalid zero quantity is intentional: it forces the second command to fail after the stock change. The normal inventory `Sell` validates quantity before starting; this miniature failure simulation is not its production input policy.


```csharp
using System;
using Microsoft.Data.Sqlite;
using var connection = new SqliteConnection("Data Source=:memory:");
connection.Open();
using (var setup = connection.CreateCommand())
{
    setup.CommandText = "CREATE TABLE Books (BookId INTEGER PRIMARY KEY, Stock INTEGER CHECK(Stock >= 0)); CREATE TABLE Sales (SaleId INTEGER PRIMARY KEY, BookId INTEGER, Quantity INTEGER CHECK(Quantity > 0)); INSERT INTO Books VALUES (1, 6);";
    setup.ExecuteNonQuery();
}
using (var transaction = connection.BeginTransaction())
{
    try
    {
        using var update = connection.CreateCommand();
        update.Transaction = transaction;
        update.CommandText = "UPDATE Books SET Stock = Stock - 2 WHERE BookId = 1 AND Stock >= 2;";
        if (update.ExecuteNonQuery() != 1) throw new InvalidOperationException("Insufficient stock.");
        using var sale = connection.CreateCommand();
        sale.Transaction = transaction;
        // Deliberate invalid quantity simulates failure of the second change.
        sale.CommandText = "INSERT INTO Sales VALUES (1, 1, 0);";
        sale.ExecuteNonQuery();
        transaction.Commit();
    }
    catch (SqliteException)
    {
        transaction.Rollback();
        Console.WriteLine("sale rejected; rolled back");
    }
}
using var check = connection.CreateCommand();
check.CommandText = "SELECT Stock FROM Books WHERE BookId = 1;";
Console.WriteLine($"stock={check.ExecuteScalar()}");
check.CommandText = "SELECT COUNT(*) FROM Sales;";
Console.WriteLine($"sales={check.ExecuteScalar()}");
```


The output is `sale rejected; rolled back`, `stock=6`, and `sales=0`. Seeing stock 4 would expose a partial change that the transaction is intended to prevent. In the full project, the forced trigger failure in check mode tests the same obligation without passing an invalid sale quantity.

**Guided prediction:** change the inserted quantity to 2 and keep `Commit`. Predict both final counts before running. Next, keep the valid insert but omit `Commit`. Explain why successful commands are not enough to establish a durable sale.


```{dropdown} Feedback and transfer
With quantity 2 and a successful commit, stock is 4 and there is one sale row; the exception message does not print. Without the commit, disposal rolls the transaction back, so stock returns to 6 and sale count to zero. Run the check *after* leaving the transaction scope, as above.

For a mapper variation, change the query ID from 1 to 999: `Read()` returns false, so there are no field reads and no book output. An empty result is not a default book with ID zero. For the update variation, change its ID to 1: its affected-row count becomes one. Use these predictions to interpret the full project's checks before extending the lab.
```


## Verify Before Extending

```{index} Verify Before Extending
```


Build verification around the contract before adapting the bookstore to another domain. Use a disposable database so the expected starting state is known. The supplied core check mode recreates its in-memory data; an ordinary file-backed run is a separate persistence check.

| Check | Action and expected evidence |
| --- | --- |
| Lookup and mapping | Read an existing BookId and compare its title, price, and stock with the seeded values. For an unknown ID, Find reports absence rather than inventing a record. |
| Parameter values | Add `Manager's SQL Guide` at 1900 cents with stock four, then look it up using the returned ID. The apostrophe remains part of the title. |
| Affected rows | Set that book's stock to six. A negative value is rejected, and an unknown ID reports false. An unrelated book's stock stays unchanged. |
| Durable state | Close and reopen the file database; the successful change remains. An in-memory database is deliberately recreated instead. |
| Optional transaction | Starting with stock six, selling two leaves four and adds one sale. Selling five afterward fails without another sale or stock change. A forced sale-insert failure must also roll back its stock decrement. |

Run `dotnet run -- --check-core` in the supplied project for the core contract. Use `dotnet run -- --check` when undertaking the optional transaction extension. Keep the expected state and failing/passing observations with the project; a green run proves only the checks actually performed. The lab applies these explanations and asks you to implement and explain the behavior.


`dotnet run -- --check` checks seed data, missing IDs, apostrophes in titles, updates, successful sales, overselling, and invalid inputs. Read the assertions before editing. Then run twice without `--check` and explain why the stored inventory persists while the check database starts fresh each time.

Keep real customer data and production credentials out of this exercise. For a semester project, document the schema, initialization, reset procedure, and tested failure cases alongside the program.


```{rubric} Footnotes
```
[^1]: Use the supplied local teaching database, rather than a production database.
